# 16. Spectrum Occupancy Prediction: Time Series vs Learning-Based Models

Implements the comparison from the paper:
**"Spectrum Occupancy Prediction for Realistic Traffic Scenarios: Time Series versus Learning-Based Models"** (Journal of Communications and Information Networks, 2018).

## Paper (adapted to our setting)
- **Time-series:** AR(p), ARIMA(p,d,q) — fit on training series; multi-step forecast (24h) from last 72 values.
- **ML:** Linear SVM (LSVM) regression, RNN (Elman: one hidden layer with recurrent context). Paper: input order 4, hidden 6 for 1-step; we use 72→24 with hidden size 6.
- **Metric in paper:** MSE (and MANE for model selection). We report MAE, RMSE, MASE to match notebooks 10–15.

## Same setup as 10–15
Data: work_dir/final, 72h→24h. Models: AR, ARIMA, LSVM, RNN (Elman), Naive. Same visuals.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
print(f'TensorFlow: {tf.__version__}')
print('statsmodels available for AR/ARIMA')

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'):
            _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU. On Apple Silicon: pip install tensorflow-metal')
USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)

## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists():
        return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try:
            dfs.append(pd.read_parquet(p))
        except Exception as e:
            print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df, test_df, lookback=72, forecast_horizon=24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    X_tr, y_tr, X_te, y_te = [], [], [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_tr.append(tr[i:i+lookback])
            y_tr.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                inp = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                h = max(0, lookback - d * forecast_horizon)
                inp = np.concatenate([tr[-h:], te[0:d*forecast_horizon]]) if h > 0 else te[d*forecast_horizon - lookback:d*forecast_horizon]
            tgt = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(inp) == lookback and len(tgt) == forecast_horizon:
                X_te.append(inp)
                y_te.append(tgt)
    if not X_tr or not X_te:
        return np.array([]), np.array([]), np.array([]), np.array([])
    return np.array(X_tr).reshape(-1, lookback, 1), np.array(y_tr), np.array(X_te).reshape(-1, lookback, 1), np.array(y_te)

In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
for band in class_options:
    if band not in train_data_by_band:
        continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(train_data_by_band[band], test_data_by_band[band], LOOKBACK, FORECAST_HORIZON)
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# One long training series for AR/ARIMA (concatenate all bands' sorted series)
train_series_list = []
for band in class_options:
    if band not in train_data_by_band:
        continue
    tr_df = train_data_by_band[band]
    ths = sorted(tr_df["threshold_dbm"].unique())
    if len(ths) > 1:
        tr_df = tr_df[tr_df["threshold_dbm"] == ths[0]]
    ser = tr_df.sort_values(["date", "hour"])["au_pct"].values
    train_series_list.append(ser)
train_series_full = np.concatenate(train_series_list).astype(np.float64)
print(f"Long training series length: {len(train_series_full)}")

## Time-series models: AR(p) and ARIMA(p,d,q)
Paper: AR order p (e.g. 6), ARIMA (1,1,1). Fit on training series; for each test sample use last 72 values and forecast 24 steps (apply params to new data then predict).

In [ ]:
AR_ORDER = 6
ARIMA_ORDER = (1, 1, 1)

# Fit AR on long training series
ar_model = AutoReg(train_series_full, lags=AR_ORDER).fit()

# Iterative 24-step forecast from 72 values using AR coefficients
def ar_forecast_24(last_72, ar_res, p):
    params = ar_res.params
    const = params[0] if len(params) > p else 0
    coefs = params[1:1+p] if len(params) > p else params[1:]
    history = list(last_72.ravel())
    preds = []
    for _ in range(FORECAST_HORIZON):
        lag_vals = history[-p:] if len(history) >= p else (history + [history[-1]] * (p - len(history)))[-p:]
        next_val = const + sum(c * lag_vals[p - 1 - j] for j, c in enumerate(coefs))
        next_val = np.clip(next_val, 0, 100)
        preds.append(next_val)
        history.append(next_val)
    return np.array(preds, dtype=np.float32)

y_pred_ar = np.zeros((len(X_test), FORECAST_HORIZON), dtype=np.float32)
for i in range(len(X_test)):
    y_pred_ar[i] = ar_forecast_24(X_test[i], ar_model, AR_ORDER)
print('AR predictions done.')

In [ ]:
# Fit ARIMA on long training series
arima_model = ARIMA(train_series_full, order=ARIMA_ORDER).fit()

# For each test sample: apply fitted ARIMA to the 72 values, then forecast 24 steps
y_pred_arima = np.zeros((len(X_test), FORECAST_HORIZON), dtype=np.float32)
for i in range(len(X_test)):
    try:
        res_new = arima_model.apply(X_test[i, :, 0].astype(np.float64))
        fcast = res_new.get_forecast(steps=FORECAST_HORIZON)
        y_pred_arima[i] = np.clip(fcast.predicted_mean.values, 0, 100)
    except Exception:
        y_pred_arima[i] = np.tile(np.mean(X_test[i, -1, 0]), FORECAST_HORIZON)
print('ARIMA predictions done.')

## ML: Linear SVM (LSVM) and RNN (Elman)

In [ ]:
# LSVM: input 72 features → output 24 (MultiOutputRegressor with LinearSVR)
X_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)
scaler_svm = MinMaxScaler(feature_range=(0, 1))
X_flat_s = scaler_svm.fit_transform(X_flat)
X_test_flat_s = scaler_svm.transform(X_test_flat)
lsvm = MultiOutputRegressor(LinearSVR(max_iter=5000, tol=1e-4, C=1.0), n_jobs=-1)
lsvm.fit(X_flat_s, y_train)
y_pred_lsvm = np.clip(lsvm.predict(X_test_flat_s), 0, 100).astype(np.float32)
print('LSVM predictions done.')

In [ ]:
# RNN (Elman): SimpleRNN with one hidden layer (6 units, paper), input (72,1) → Dense(24)
BATCH = 128 if USE_GPU else 32
EPOCHS = 50
scaler_rnn_x = MinMaxScaler(feature_range=(0, 1))
scaler_rnn_y = MinMaxScaler(feature_range=(0, 1))
X_train_rnn = scaler_rnn_x.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
X_test_rnn = scaler_rnn_x.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
y_train_rnn = scaler_rnn_y.fit_transform(y_train.reshape(-1, 1)).reshape(y_train.shape)

rnn_elman = keras.Sequential([
    layers.SimpleRNN(6, activation='tanh', input_shape=(LOOKBACK, 1)),
    layers.Dense(FORECAST_HORIZON, activation='linear'),
])
rnn_elman.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])
rnn_elman.fit(X_train_rnn, y_train_rnn, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)
y_pred_rnn_s = rnn_elman.predict(X_test_rnn, verbose=0)
y_pred_rnn = scaler_rnn_y.inverse_transform(y_pred_rnn_s.reshape(-1, 1)).reshape(y_test.shape)
y_pred_rnn = np.clip(y_pred_rnn, 0, 100).astype(np.float32)
print('RNN (Elman) predictions done.')

In [ ]:
# Naive baseline and metric helpers
def naive_predictor(X, horizon):
    last = X[:, -1, 0]
    return np.tile(last.reshape(-1, 1), (1, horizon))
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

In [ ]:
# Results table (same format as notebook 10)
mae_ar = calculate_mae(y_test, y_pred_ar)
rmse_ar = calculate_rmse(y_test, y_pred_ar)
mase_ar = calculate_mase(y_test, y_pred_ar, y_train)
mae_arima = calculate_mae(y_test, y_pred_arima)
rmse_arima = calculate_rmse(y_test, y_pred_arima)
mase_arima = calculate_mase(y_test, y_pred_arima, y_train)
mae_lsvm = calculate_mae(y_test, y_pred_lsvm)
rmse_lsvm = calculate_rmse(y_test, y_pred_lsvm)
mase_lsvm = calculate_mase(y_test, y_pred_lsvm, y_train)
mae_rnn = calculate_mae(y_test, y_pred_rnn)
rmse_rnn = calculate_rmse(y_test, y_pred_rnn)
mase_rnn = calculate_mase(y_test, y_pred_rnn, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "AR", "MAE": mae_ar, "RMSE": rmse_ar, "MASE": mase_ar},
    {"Model": "ARIMA", "MAE": mae_arima, "RMSE": rmse_arima, "MASE": mase_arima},
    {"Model": "LSVM", "MAE": mae_lsvm, "RMSE": rmse_lsvm, "MASE": mase_lsvm},
    {"Model": "RNN (Elman)", "MAE": mae_rnn, "RMSE": rmse_rnn, "MASE": mase_rnn},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY (same format as notebook 10)")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(f"\n{results_df.to_string(index=False)}")
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
# 1. Bar charts: MAE, RMSE, MASE
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.suptitle('Next-day prediction: Time Series vs ML (paper comparison)', y=1.02, fontsize=12)
plt.show()

In [ ]:
# 2. Improvement over Naive Baseline (%)
naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive Baseline (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Predicted vs Actual (up to 3 samples) + mean profile — best ML (RNN) vs best TS (AR or ARIMA) vs Naive
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_rnn[i], '-', linewidth=1.6, label='RNN (Elman)')
    ax.plot(hours, y_pred_ar[i], '-', linewidth=1.2, label='AR', alpha=0.8)
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted (solid) vs Actual (dashed)')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_rnn.mean(axis=0), '-', linewidth=1.6, label='RNN (mean)')
ax.plot(hours, y_pred_ar.mean(axis=0), '-', linewidth=1.2, label='AR (mean)', alpha=0.8)
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4. Per-hour MAE (RNN vs AR vs Naive)
mae_per_hour_rnn = np.abs(y_test - y_pred_rnn).mean(axis=0)
mae_per_hour_ar = np.abs(y_test - y_pred_ar).mean(axis=0)
mae_per_hour_naive = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour_rnn, '-o', label='RNN (Elman)', markersize=4)
ax.plot(hours, mae_per_hour_ar, '-o', label='AR', markersize=4, alpha=0.8)
ax.plot(hours, mae_per_hour_naive, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5. Residuals: best model (lowest MAE among non-Naive) vs Naive
best_non_naive = results_df[results_df['Model'] != 'Naive Baseline'].loc[results_df[results_df['Model'] != 'Naive Baseline']['MAE'].idxmin(), 'Model']
pred_best = {'AR': y_pred_ar, 'ARIMA': y_pred_arima, 'LSVM': y_pred_lsvm, 'RNN (Elman)': y_pred_rnn}[best_non_naive]
residuals_best = (y_test - pred_best).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_best, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Residuals: {best_non_naive}')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive Baseline')
plt.suptitle('Residual distribution (centered at 0 is ideal)', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): {best_non_naive} = {residuals_best.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         {best_non_naive} = {residuals_best.std():.4f}, Naive = {residuals_naive.std():.4f}')

### Key insights

- **Paper:** Time-series (AR, ARIMA) work well for stationary/periodic traffic; ML (LSVM, RNN) can outperform for non-stationary data. RNN (Elman) generally gave best accuracy in the paper.
- **Improvement over naive:** Positive % means the model beats the last-value baseline.
- **Mean profile / per-hour MAE / residuals:** Same interpretation as notebooks 10–15.

In [ ]:
# One-line summary (same format as notebook 10)
subset = results_df[results_df['Model'] != 'Naive Baseline']
best_row = results_df.loc[subset['MAE'].idxmin()]
best_name = best_row['Model']
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"Best model: {best_name} (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}).")
print(f"Improvement over Naive: MAE {imp_mae:+.1f}%.")